# Bhojpuri SUMMARY — ZERO-SHOT — `indicbart_pre`

**Model:** `ai4bharat/IndicBART` · **Test split:** `bho_test.csv` (218 rows) · **Shots:** 0

Your summary prompt is reproduced verbatim with only the exemplar block removed
(prompt sha1 of the string values = `66e49f74bc05f56d`). Affect-preservation clauses retained.

| setting | value | matches fine-tuned IndicBART |
|---|---|---|
| eval set | 218-row test split | yes |
| decoding | beams=4, len_pen=1.0, no_repeat=3 | yes |
| `max_new_tokens` | 200 | yes |
| precision | bfloat16 (no bitsandbytes) | — |
| post-processing | cleanup only, no length cap | yes (none) |
| empty output | scored 0, never dropped | yes |

**Before running:** Kernel → Restart, then Run All from the top. Cell 0 must run
before anything imports `torch`.


In [1]:
# ============================================================
# CELL 0 — environment. RUN FIRST, after Kernel -> Restart.
# ------------------------------------------------------------
# This node has no C headers (stdlib.h absent), so Triton can never JIT-compile.
# We therefore stay on bf16 (no bitsandbytes) and disable every compilation
# path, letting any compile attempt degrade to eager execution.
# ============================================================
import os, shutil
for _k, _v in {"TORCHINDUCTOR_DISABLE":"1", "TORCHDYNAMO_DISABLE":"1",
               "TORCH_COMPILE_DISABLE":"1", "BITSANDBYTES_NOWELCOME":"1",
               "TOKENIZERS_PARALLELISM":"false", "HF_HUB_DISABLE_XET":"1",
               "HF_HUB_DOWNLOAD_TIMEOUT":"60"}.items():
    os.environ[_k] = _v
for _v in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    _cur = os.environ.get(_v, "")
    if "/lib64" not in _cur.split(":"):
        os.environ[_v] = f"/lib64:{_cur}" if _cur else "/lib64"
shutil.rmtree(os.path.expanduser("~/.triton/cache"), ignore_errors=True)

import torch
torch._dynamo.config.suppress_errors = True
_free, _total = torch.cuda.mem_get_info()
VRAM_GB = _free / (1024 ** 3)
print(f"torch {torch.__version__} | {_total/1e9:.0f} GB total, {_free/1e9:.1f} GB FREE")
if VRAM_GB < 25:
    print("!" * 60)
    print(f"Only {VRAM_GB:.1f} GB free. Stale kernels are the usual cause:")
    print("   Kernel -> Shut Down All Kernels, reopen, rerun.   Check: !nvidia-smi")
    print("!" * 60)


torch 2.14.0+cu126 | 25 GB total, 24.6 GB FREE
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Only 22.9 GB free. Stale kernels are the usual cause:
   Kernel -> Shut Down All Kernels, reopen, rerun.   Check: !nvidia-smi
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
# ============================================================
# CELL 1 — configuration (nothing to edit)
# ============================================================
MODEL_KEY  = "indicbart_pre"
MODEL_NAME = "ai4bharat/IndicBART"
ARCH       = "seq2seq"         # causal | seq2seq
FMT        = "none"   # chat-template wrapper — the ONLY per-model difference
GATED      = False

LANG, TASK = "bho", "summary"
TEST_CSV, EXPECT_N = "bho_test.csv", 218

SHOTS            = 0
NUM_BEAMS        = 4
LENGTH_PENALTY   = 1.0
NO_REPEAT_NGRAM  = 3
DO_SAMPLE        = False
MAX_NEW_TOKENS   = 200          # same as fine-tuned IndicBART summary
MAX_INPUT_TOKENS = 1024
USE_4BIT         = False        # bf16: bitsandbytes needs a compiler this node lacks
BATCH_SIZE       = 2            # summaries are longer than headlines
SEED             = 42
FRESH_START      = True

NEED_GB = 3
print(f"RUN {MODEL_KEY} | {LANG}/{TASK} | beams={NUM_BEAMS} max_new={MAX_NEW_TOKENS} "
      f"bf16 | shots={SHOTS}")
if VRAM_GB < NEED_GB:
    raise SystemExit(f"{MODEL_KEY} needs ~{NEED_GB} GB in bf16, only {VRAM_GB:.1f} GB free. "
                     f"Free the GPU and rerun.")

RUN_TAG        = f"{MODEL_KEY}_{LANG}_{TASK}_zeroshot"
CHECKPOINT_CSV = f"ckpt_{RUN_TAG}.csv"
FINAL_CSV      = f"gen_{RUN_TAG}.csv"
PERITEM_CSV    = f"per_item_{RUN_TAG}.csv"
METRICS_JSON   = f"metrics_{RUN_TAG}.json"
MANIFEST_JSON  = f"manifest_{RUN_TAG}.json"


RUN indicbart_pre | bho/summary | beams=4 max_new=200 bf16 | shots=0


In [4]:
# ============================================================
# CELL 2 — imports, seeding, GPU guard, HF auth (gated only)
# ============================================================
import re, gc, json, time, glob, random, hashlib, unicodedata
import numpy as np, pandas as pd

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "No GPU — a CPU fallback would take days."
print("device:", torch.cuda.get_device_name(0))

if GATED:
    from huggingface_hub import login, whoami, hf_hub_download
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("HF token (hidden): ").strip()
        os.environ["HF_TOKEN"] = tok
    login(tok, add_to_git_credential=False)
    print("logged in as:", whoami()["name"])
    try:
        hf_hub_download(MODEL_NAME, "config.json"); print("access OK")
    except Exception as e:
        raise SystemExit(f"No access to {MODEL_NAME}: {str(e)[:150]}\n"
                         f"Check https://huggingface.co/settings/gated-repos")


device: NVIDIA RTX A5000


In [5]:
# ============================================================
# CELL 3 — load the TEST SPLIT (self-healing)
# ============================================================
TEXT_COL, REF_COL, GEN_COL = "article_text", "summary", "generated"

cands = [TEST_CSV] + sorted(set(glob.glob("*test*.csv")))
path = next((p for p in cands if os.path.exists(p)), None)
if path is None:
    raise SystemExit(f"No test CSV in {os.getcwd()}. Expected {TEST_CSV}.")
if path != TEST_CSV:
    print(f"NOTE: using '{path}' (did not find {TEST_CSV})")

df = pd.read_csv(path, dtype=str, keep_default_na=False)
alias = {"article_text":["article_text","text","article","body","content"],
         "summary":["summary","gold_summary","reference_summary"],
         "headline":["headline","article_headline","title"]}
low = {c.strip().lower(): c for c in df.columns}
for want, alts in alias.items():
    if want not in df.columns:
        hit = next((low[a] for a in alts if a in low), None)
        if hit: df = df.rename(columns={hit: want}); print(f"  renamed {hit} -> {want}")
for c in (TEXT_COL, REF_COL):
    assert c in df.columns, f"missing '{c}'. Have: {list(df.columns)}"
    df[c] = df[c].astype(str).str.strip()
if "row_key" not in df.columns:
    df["row_key"] = df[TEXT_COL].map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:16])
if not df["row_key"].is_unique:
    df["row_key"] = [f"{k}_{i}" for i, k in enumerate(df["row_key"])]

if len(df) != EXPECT_N:
    print(f"\n!!! {len(df)} rows, expected {EXPECT_N} (Table 6). Scores will NOT be "
          f"comparable to fine-tuned IndicBART. Continuing; true n is recorded.\n")
    EXPECT_N = len(df)
else:
    print(f"OK: {len(df)} rows matches Table 6")

df[GEN_COL] = ""
w = df[REF_COL].str.split().str.len()
print(f"gold summary words: mean={w.mean():.1f} median={w.median():.0f} "
      f"min={w.min()} max={w.max()}")


OK: 218 rows matches Table 6
gold summary words: mean=80.6 median=81 min=42 max=123


In [6]:
# ============================================================
# CELL 4 — article cleaning
# ============================================================
_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u200e\u200f\ufeff"), None)
def clean_article(t):
    s = "" if t is None else str(t)
    if s.strip().lower() in {"nan","none","null"}: return ""
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\t\n ")
    return re.sub(r"\s+", " ", s).strip()
df["_clean"] = df[TEXT_COL].map(clean_article)
print("empty articles:", int((df["_clean"] == "").sum()))


empty articles: 0


In [7]:
# ============================================================
# CELL 5 — PROMPT (your summary prompt, verbatim; exemplar block REMOVED)
# ============================================================
SYSTEM_PROMPT = 'रउआ एगो भोजपुरी समाचार सारांश लेखक बानी। रउआ हमेशा शुद्ध भोजपुरी में ही लिखेनी, हिंदी में कबो ना लिखेनी। रउआ लेख के मूल भाव आ भावना के जस के तस राखेनी।'

def _instruction_block():
    return 'नीचे दिहल भोजपुरी समाचार लेख के भोजपुरी भाषा में 80 से 100 शब्दन के सारांश लिखीं। सारांश खाली शुद्ध भोजपुरी में होखे के चाहीं — हिंदी में मत लिखीं। भोजपुरी के शब्द इस्तेमाल करीं जइसे: बा, बाड़ें, बानी, गइल, कइलस, रहल, होखे, चाहीं, रउआ, हमनी, ओकर, एकर।\n\nअउरी जरूरी बात:\n- सारांश 80 से 100 शब्दन के बीच में ही होखे। 100 शब्दन से जादे मत लिखीं।\n- लेख के भाव (खुशी, दुख, चिंता, गुस्सा, आस, निराशा, उत्साह) आ स्वर के जस के तस राखीं। लेख के भाव के हल्का भा साधारण मत करीं।\n- लेख में जवन नाम, जगह, संख्या आ तारीख बा, ओकरा के सारांश में शामिल करीं।\n- आपन ओर से कवनो नया जानकारी भा राय मत जोड़ीं। जवन लेख में बा, खाली ओकरे आधार पर लिखीं।\n- खाली एक बेर में एक साथ सारांश लिखीं। कवनो शीर्षक, ‘सारांश:’ जइसन शब्द, बुलेट भा नंबर मत जोड़ीं।'

def build_user_prompt(cleaned):
    """ZERO-SHOT. The original's 'एगो उदाहरण' exemplar block is deliberately absent."""
    return (_instruction_block()
            + f"\n\nलेख:\n{cleaned}\n\n"
            + "भोजपुरी सारांश (80-100 शब्द, खाली भोजपुरी में):")

def format_input(cleaned):
    u = build_user_prompt(cleaned)
    if FMT == "airavata":        # <|system|>/<|user|>/<|assistant|>, add_bos=False as in yours
        return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{u}\n<|assistant|>\n"
    if FMT == "plain":           # base LMs without a chat template
        return f"{SYSTEM_PROMPT}\n\n{u}"
    if FMT == "chat_sys":        # Llama-3.1
        return tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":u}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "chat_nosys":      # Gemma-2 has no system role
        return tokenizer.apply_chat_template(
            [{"role":"user","content":SYSTEM_PROMPT + "\n\n" + u}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "none":            # IndicBART: no instruction, language tag only
        return f"<2hi> {cleaned} </s>"
    raise ValueError(FMT)

PROMPT_SHA1 = hashlib.sha1((SYSTEM_PROMPT + _instruction_block()).encode()).hexdigest()[:16]
print("prompt sha1 =", PROMPT_SHA1, "  (expected 66e49f74bc05f56d)")
assert PROMPT_SHA1 == "66e49f74bc05f56d", "prompt text drifted from the verified version"
print("SHOTS = 0 -> no exemplar")


prompt sha1 = 66e49f74bc05f56d   (expected 66e49f74bc05f56d)
SHOTS = 0 -> no exemplar


In [8]:
# ============================================================
# CELL 6 — load model (bf16, eager attention, no bitsandbytes)
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, MBartForConditionalGeneration

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if ARCH == "seq2seq":
    model = MBartForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float32).cuda()
else:
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True, attn_implementation="eager")
model.eval()
print(f"loaded {MODEL_NAME} | {torch.cuda.memory_allocated()/1e9:.1f} GB")


Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

loaded ai4bharat/IndicBART | 1.0 GB


In [9]:
# ============================================================
# CELL 7 — post-processing (NO length cap) + generation
# ============================================================
_MARKERS = ("<|assistant|>","<|user|>","<|system|>","<|end|>","<|eot_id|>",
            "<|start_header_id|>","<|end_header_id|>","</s>","<s>","[/INST]",
            "<end_of_turn>","<start_of_turn>","<2hi>")

def postprocess(t):
    if not t: return ""
    s = str(t)
    for m in _MARKERS: s = s.replace(m, " ")
    s = re.sub(r"^\s*(भोजपुरी\s*)?(सारांश|summary)\s*[:：\-]\s*", "", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip().strip(' "\u201c\u201d')

NTOK = {}   # generated-token count per row, to measure truncation at the cap

@torch.no_grad()
def generate_batch(cleans):
    enc = tokenizer([format_input(c) for c in cleans], return_tensors="pt",
                    padding=True, truncation=True, max_length=MAX_INPUT_TOKENS,
                    add_special_tokens=(ARCH != "seq2seq")).to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS,
              length_penalty=LENGTH_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM,
              do_sample=DO_SAMPLE, early_stopping=True)
    if ARCH == "seq2seq":
        kw["decoder_start_token_id"] = tokenizer.convert_tokens_to_ids("<2hi>")
        out = model.generate(**enc, **kw)
        toks = out
    else:
        kw["pad_token_id"] = tokenizer.pad_token_id
        out = model.generate(**enc, **kw)
        toks = out[:, enc["input_ids"].shape[1]:]
    pad = tokenizer.pad_token_id
    lens = [int((row != pad).sum()) for row in toks]
    dec = tokenizer.batch_decode(toks, skip_special_tokens=True)
    return [postprocess(t) for t in dec], lens

t0 = time.time()
g, l = generate_batch(df["_clean"].head(2).tolist())
for i, (x, n) in enumerate(zip(g, l)):
    print(f"SMOKE {i}: [{n} tok, {len(x.split())} w] {x[:120]}")
est = (time.time() - t0) / 2 * len(df) / 60
print(f"est. ~{est:.0f} min for {len(df)} rows")


SMOKE 0: [201 tok, 138 w] न्यूज सेंट्रल डेस्क: बैंकॉक से साउथ कोरिया आ रहल जेजू एयर के एगो फ्लाइट अतवार के मुआन इंटरनेशनल एयरपोर्ट पs क्रैश हो गइल
SMOKE 1: [201 tok, 135 w] नई Champions Trophy: अगिला साल होखे वाला चैंपियंस ट्रॉफी टूर्नामेंट खातिर अभी ले वेन्यू तय नइखे भइल। पाकिस्तान क्रिकेट ब
est. ~6 min for 218 rows


In [10]:
# ============================================================
# CELL 8 — generation loop (fresh start, checkpointed, OOM-safe)
# ============================================================
if os.path.exists(CHECKPOINT_CSV):
    if FRESH_START:
        os.rename(CHECKPOINT_CSV, f"{CHECKPOINT_CSV}.bak.{int(time.time())}")
        print("old checkpoint moved aside")
    else:
        ck = pd.read_csv(CHECKPOINT_CSV, dtype=str, keep_default_na=False)
        m = dict(zip(ck["row_key"], ck[GEN_COL])); df[GEN_COL] = df["row_key"].map(lambda k: m.get(k, ""))

todo = df.index[df[GEN_COL].str.strip() == ""].tolist()
print(f"generating {len(todo)}/{len(df)}")
t0 = time.time()
for s in range(0, len(todo), BATCH_SIZE):
    idx = todo[s:s+BATCH_SIZE]
    batch = df.loc[idx, "_clean"].tolist()
    try:
        outs, lens = generate_batch(batch)
    except Exception as e:
        print(f"  [batch {s}: {type(e).__name__}] retrying singly")
        torch.cuda.empty_cache(); outs, lens = [], []
        for a in batch:
            try:  o, n = generate_batch([a]); outs.append(o[0]); lens.append(n[0])
            except Exception: outs.append(""); lens.append(0)
    for i, o, n in zip(idx, outs, lens):
        df.at[i, GEN_COL] = o; NTOK[df.at[i, "row_key"]] = n
    df[["row_key", GEN_COL]].to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")
    if (s // BATCH_SIZE) % 10 == 0 or s + BATCH_SIZE >= len(todo):
        d = min(s + BATCH_SIZE, len(todo)); el = time.time() - t0
        print(f"  {d}/{len(todo)} | {el/60:.1f} min | ~{el/max(d,1)*(len(todo)-d)/60:.1f} min left")

print(f"done in {(time.time()-t0)/60:.1f} min | empty "
      f"{int((df[GEN_COL].str.strip()=='').sum())}/{len(df)}")
df.drop(columns=["_clean"]).to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
for i in df.index[:3]:
    print(f"\nGOLD: {df.at[i, REF_COL][:140]}\nGEN : {df.at[i, GEN_COL][:140]}")


old checkpoint moved aside
generating 218/218


  2/218 | 0.0 min | ~4.5 min left


  22/218 | 0.4 min | ~3.7 min left


  42/218 | 0.8 min | ~3.2 min left


  62/218 | 1.1 min | ~2.8 min left


...

  142/218 | 2.5 min | ~1.3 min left


  162/218 | 2.8 min | ~1.0 min left


  182/218 | 3.1 min | ~0.6 min left


  202/218 | 3.4 min | ~0.3 min left


  218/218 | 3.6 min | ~0.0 min left
done in 3.6 min | empty 0/218

GOLD: बैंकॉक से साउथ कोरिया आ रहल जेजू एयर के एगो फ्लाइट मुआन इंटरनेशनल एयरपोर्ट पs क्रैश हो गइल। प्लेन में सवार 181 लोगन में से 85 लोगन के शव बरा
GEN : न्यूज सेंट्रल डेस्क: बैंकॉक से साउथ कोरिया आ रहल जेजू एयर के एगो फ्लाइट अतवार के मुआन इंटरनेशनल एयरपोर्ट पs क्रैश हो गइल। कोरिया टाइम्स के र

GOLD: अगिला साल होखे वाला चैंपियंस ट्रॉफी खातिर अभी ले वेन्यू तय नइखे भइल। पाकिस्तान क्रिकेट बोर्ड आधिकारिक मेजबान बा, बाकिर बीसीसीआई के टीम इंडिय
GEN : नई Champions Trophy: अगिला साल होखे वाला चैंपियंस ट्रॉफी टूर्नामेंट खातिर अभी ले वेन्यू तय नइखे भइल। पाकिस्तान क्रिकेट बोर्ड एकर आधिकारिक मे

GOLD: भारतीय महिला हॉकी टीम के दिग्गज वंदना कटारिया मंगल के दिने अंतर्राष्ट्रीय हॉकी से संन्यास लेबे के एलान कइली। उs 15 साल तक भारतीय हॉकी में यो
GEN : भारतीय महिला हॉकी टीम के दिग्गज वंदना कटारिया मंगल के दिने अंतर्राष्ट्रीय हॉकी से संन्यास लेबे के एलान कइली. 15 साल तक भारतीय हॉकी में योगदा


In [11]:
# ============================================================
# CELL 9 — STANDARDIZED EVALUATION (identical across all models)
# ============================================================
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

class _WS:
    def tokenize(self, t): return str(t).split()
sc = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=False, tokenizer=_WS())
def ci(v, n=1000):
    rng = np.random.default_rng(SEED); a = np.asarray(v, float)
    b = [a[rng.integers(0, len(a), len(a))].mean() for _ in range(n)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]

refs = df[REF_COL].fillna("").astype(str).str.strip().tolist()
hyps = df[GEN_COL].fillna("").astype(str).str.strip().tolist()
assert len(refs) == len(hyps) == EXPECT_N

per = [sc.score(r, h) for r, h in zip(refs, hyps)]          # empty -> 0, retained
r1 = [p["rouge1"].fmeasure for p in per]; r2 = [p["rouge2"].fmeasure for p in per]
rl = [p["rougeL"].fmeasure for p in per]
bl = [sacrebleu.sentence_bleu(h, [r], smooth_method="exp", tokenize="intl").score/100
      for h, r in zip(hyps, refs)]
cb = sacrebleu.corpus_bleu(hyps, [refs], tokenize="intl")
_, _, F = bert_score_fn(hyps, refs, lang="hi", model_type="bert-base-multilingual-cased",
                        batch_size=8, verbose=False)
bs = F.tolist(); lens = [len(h.split()) for h in hyps]
ntok = [NTOK.get(k, 0) for k in df["row_key"]]
hit_cap = float(np.mean([n >= MAX_NEW_TOKENS - 1 for n in ntok])) if NTOK else float("nan")

M = {"run":RUN_TAG, "model":MODEL_NAME, "lang":LANG, "task":TASK, "shots":SHOTS,
     "n":EXPECT_N, "valid_rate":float(np.mean([bool(h) for h in hyps])),
     "bleu_sentence":float(np.mean(bl)), "bleu_sentence_ci":ci(bl),
     "rouge1":float(np.mean(r1)), "rouge1_ci":ci(r1),
     "rouge2":float(np.mean(r2)), "rouge2_ci":ci(r2),
     "rougeL":float(np.mean(rl)), "rougeL_ci":ci(rl),
     "bertscore_f1":float(np.mean(bs)), "bertscore_f1_ci":ci(bs),
     "brevity_penalty":float(cb.bp), "bleu_corpus":float(cb.score/100),
     "len_mean":float(np.mean(lens)), "len_median":float(np.median(lens)),
     "len_std":float(np.std(lens)),
     "pct_80_100w":float(np.mean([80 <= l <= 100 for l in lens])),
     "pct_hit_token_cap":hit_cap}

print(f"\n{'='*62}\n{RUN_TAG}  n={M['n']}  valid={M['valid_rate']:.1%}\n{'='*62}")
for k in ("bleu_sentence","rouge1","rouge2","rougeL","bertscore_f1"):
    lo, hi = M[k+"_ci"]; print(f"  {k:<15} {M[k]:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  brevity_penalty {M['brevity_penalty']:.4f}")
print(f"  length  mean={M['len_mean']:.1f} med={M['len_median']:.0f} "
      f"in 80-100w={M['pct_80_100w']:.1%}")
print(f"  hit {MAX_NEW_TOKENS}-token cap: {hit_cap:.1%}   "
      f"<- if high, outputs were truncated; report it")

pd.DataFrame({"row_key":df["row_key"], "ref":refs, "hyp":hyps, "rouge1":r1, "rouge2":r2,
              "rougeL":rl, "bleu":bl, "bertscore":bs, "len":lens, "ntok":ntok}
            ).to_csv(PERITEM_CSV, index=False, encoding="utf-8-sig")
json.dump(M, open(METRICS_JSON, "w"), indent=2, ensure_ascii=False)
print(f"\nsaved {PERITEM_CSV}\nsaved {METRICS_JSON}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



indicbart_pre_bho_summary_zeroshot  n=218  valid=100.0%
  bleu_sentence   0.2540  95% CI [0.2404, 0.2682]
  rouge1          0.5256  95% CI [0.5130, 0.5382]
  rouge2          0.3420  95% CI [0.3265, 0.3585]
  rougeL          0.4286  95% CI [0.4130, 0.4441]
  bertscore_f1    0.7815  95% CI [0.7759, 0.7869]
  brevity_penalty 1.0000
  length  mean=132.8 med=136 in 80-100w=1.8%
  hit 200-token cap: 99.1%   <- if high, outputs were truncated; report it

saved per_item_indicbart_pre_bho_summary_zeroshot.csv
saved metrics_indicbart_pre_bho_summary_zeroshot.json


In [12]:
# ============================================================
# CELL 10 — protocol manifest (Section V-B) + failure-mode sample + cleanup
# ============================================================
json.dump({"run":RUN_TAG, "model_name":MODEL_NAME, "arch":ARCH, "chat_format":FMT,
           "language":LANG, "task":TASK, "eval_set":TEST_CSV, "n_eval":EXPECT_N,
           "split_seed":SEED, "reference_column":REF_COL, "shots":SHOTS, "exemplar":None,
           "prompt_sha1":PROMPT_SHA1, "affect_clause_in_prompt":True,
           "decoding":{"num_beams":NUM_BEAMS, "length_penalty":LENGTH_PENALTY,
                       "no_repeat_ngram_size":NO_REPEAT_NGRAM, "do_sample":DO_SAMPLE,
                       "max_new_tokens":MAX_NEW_TOKENS, "max_input_tokens":MAX_INPUT_TOKENS},
           "precision":"float32" if ARCH=="seq2seq" else "bfloat16",
           "attn_implementation":None if ARCH=="seq2seq" else "eager",
           "postprocessing":"marker + label cleanup only; NO length cap",
           "empty_output_policy":"scored as 0, never dropped",
           "metrics":{"rouge":"word-level whitespace, per-instance F1",
                      "bleu":"sentence-level, smooth=exp, tokenize=intl, /100",
                      "bertscore":"bert-base-multilingual-cased, lang=hi",
                      "ci":"1000-sample bootstrap, 95%"}},
          open(MANIFEST_JSON, "w"), indent=2, ensure_ascii=False)
print("saved", MANIFEST_JSON)

print("\n--- 15 outputs for failure-mode notes (Hindi? repetition? echo? truncated?) ---")
for i in df.index[:15]:
    h = df.at[i, GEN_COL]; k = df.at[i, "row_key"]
    print(f"[{len(h.split()):>3}w {NTOK.get(k,0):>3}tok] {h[:110] if h else '<EMPTY>'}")

del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print("\nGPU released.")


saved manifest_indicbart_pre_bho_summary_zeroshot.json

--- 15 outputs for failure-mode notes (Hindi? repetition? echo? truncated?) ---
[138w 201tok] न्यूज सेंट्रल डेस्क: बैंकॉक से साउथ कोरिया आ रहल जेजू एयर के एगो फ्लाइट अतवार के मुआन इंटरनेशनल एयरपोर्ट पs क्
[135w 201tok] नई Champions Trophy: अगिला साल होखे वाला चैंपियंस ट्रॉफी टूर्नामेंट खातिर अभी ले वेन्यू तय नइखे भइल। पाकिस्तान
[141w 201tok] भारतीय महिला हॉकी टीम के दिग्गज वंदना कटारिया मंगल के दिने अंतर्राष्ट्रीय हॉकी से संन्यास लेबे के एलान कइली. 1
[144w 201tok] यशस्वी जायसवाल शानदार फॉर्म में बाड़े अवुरी उनुका बल्लेबाजी से रन भी आईल बा। आंकड़ा एकर गवाही दे रहल बा. आईपीए
[127w 201tok] भारत प्रधानमंत्री नरेंद्र मोदी डिजिटल कंटेंट क्रिएटर्स के बाद बियफे के देश के कुछ शीर्ष गेमर्स सन से मुलाकात क
[131w 201tok] टेक डेस्क: एपल सोमार के आपन वार्षिक वर्ल्ड वाइड डेवेलपर कॉन्फ्रेंस (WWDC) 2025 के दौरान अपना सब ऑपरेटिंग सिस्ट
[ 60w 201tok] Lenovo IdeaPad Slim 3 के 34,990 रुपया के कीमत पर खरीदल जा सकsता। लैपटॉप में 15.6 इंच के फुलएचडी प्लस


GPU released.
